In [13]:
# ==================================================================================
# 🏆 TRACK B WINNING SOLUTION: ROBUST PATH FINDING + PRE-TRAINING
# ==================================================================================
# !pip install transformers datasets torch torchvision tqdm pandas scikit-learn -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm

# ==========================================
# 1. ROBUST CONFIGURATION
# ==========================================
class Config:
    MODEL_NAME = "distilbert-base-uncased"
    STATE_DIM = 256
    CHUNK_SIZE = 128
    PRETRAIN_EPOCHS = 3
    FINETUNE_EPOCHS = 10
    PRETRAIN_LR = 1e-3
    FINETUNE_LR = 1e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 🔍 AUTO-DETECT DATA PATH
    # We check common locations for 'train.csv'
    DATA_DIR = None
    possible_paths = [
        "kdshdata",                     # Local folder
        "/kaggle/input/kdshdata",       # Standard Kaggle dataset
        "/kaggle/input/kdshdata/kdshdata", # Nested Kaggle dataset
        "/kaggle/input",                # Root input
        "../input/kdshdata",            # Alternative input
        "."                             # Current directory
    ]
    
    print(f"Searching for data...")
    for path in possible_paths:
        if os.path.exists(path) and os.path.exists(os.path.join(path, "train.csv")):
            DATA_DIR = path
            print(f"✅ FOUND DATA AT: {DATA_DIR}")
            break
            
    if DATA_DIR is None:
        print("❌ ERROR: Could not find 'train.csv' in any common paths.")
        print("Listing /kaggle/input to help you debug:")
        if os.path.exists("/kaggle/input"):
            print(os.listdir("/kaggle/input"))
        else:
            print("/kaggle/input does not exist.")
        # Fallback to avoid crashing immediately (will crash later if not fixed)
        DATA_DIR = "kdshdata"

print(f"Running on Device: {Config.DEVICE}")

# ==========================================
# 2. MODEL ARCHITECTURE (BDH Recurrent)
# ==========================================
class BDHRecurrentCell(nn.Module):
    def __init__(self, input_dim, state_dim):
        super().__init__()
        self.state_dim = state_dim
        self.W = nn.Parameter(torch.randn(state_dim, input_dim) * 0.02)
        self.gate = nn.Sequential(
            nn.Linear(input_dim + state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        self.ln = nn.LayerNorm(state_dim)

    def forward(self, x, h):
        if h is None:
            h = torch.zeros(x.size(0), self.state_dim).to(x.device)
        cand = F.linear(x, self.W)
        combined = torch.cat([x, h], dim=1)
        g = self.gate(combined)
        h_new = self.ln(F.tanh(h + g * cand))
        return h_new, g

class NarrativeModel(nn.Module):
    def __init__(self, model_name, state_dim=256):
        super().__init__()
        self.llm = AutoModel.from_pretrained(model_name)
        for p in self.llm.parameters(): p.requires_grad = False
        self.adapter = nn.Linear(self.llm.config.hidden_size, state_dim)
        self.cell = BDHRecurrentCell(state_dim, state_dim)
        self.classifier = nn.Linear(state_dim, 2) 

    def forward_pretrain(self, ids_t, mask_t, ids_next, mask_next):
        with torch.no_grad():
            emb_t = self.llm(ids_t, mask_t).last_hidden_state[:, 0]
        feat_t = F.relu(self.adapter(emb_t))
        with torch.no_grad():
            emb_next = self.llm(ids_next, mask_next).last_hidden_state[:, 0]
        target_feat = F.relu(self.adapter(emb_next))
        h_prev = torch.zeros(ids_t.size(0), Config.STATE_DIM).to(ids_t.device)
        h_new, _ = self.cell(feat_t, h_prev)
        return h_new, target_feat

    def forward_finetune(self, input_ids_chunks, mask_chunks):
        batch_size, num_chunks, seq_len = input_ids_chunks.shape
        h = None
        for t in range(num_chunks):
            ids = input_ids_chunks[:, t, :]
            mask = mask_chunks[:, t, :]
            with torch.no_grad():
                emb = self.llm(ids, mask).last_hidden_state[:, 0]
            feat = F.relu(self.adapter(emb))
            h, _ = self.cell(feat, h)
        return self.classifier(h)

# ==========================================
# 3. DATASETS
# ==========================================
class NarrativePretrainingDataset(Dataset):
    def __init__(self, file_paths, tokenizer, chunk_size=128):
        self.samples = []
        for fpath in file_paths:
            if not os.path.exists(fpath): continue
            try:
                with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
                tokens = tokenizer(text, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
                for i in range(0, len(tokens) - chunk_size * 2, chunk_size):
                    self.samples.append((
                        tokens[i : i + chunk_size],
                        tokens[i + chunk_size : i + 2*chunk_size]
                    ))
            except Exception as e:
                print(f"Error reading {fpath}: {e}")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        c_t, c_next = self.samples[idx]
        return {'ids_t': c_t, 'mask_t': torch.ones_like(c_t), 'ids_next': c_next, 'mask_next': torch.ones_like(c_next)}

class LabeledNarrativeDataset(Dataset):
    def __init__(self, df, tokenizer, chunk_size=128, max_chunks=16):
        self.df = df
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_chunks = max_chunks
        self.label_map = {'consistent': 1, 'contradict': 0}
        
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f"{row['book_name']} [SEP] {row['char']} [SEP] {row['content']}"
        tokens = self.tokenizer(text, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        final_ids = torch.zeros(self.max_chunks, self.chunk_size, dtype=torch.long)
        final_mask = torch.zeros(self.max_chunks, self.chunk_size, dtype=torch.long)
        chunks = tokens.split(self.chunk_size)
        for i, chunk in enumerate(chunks[:self.max_chunks]):
            l = len(chunk)
            final_ids[i, :l] = chunk
            final_mask[i, :l] = 1
        label = self.label_map.get(row['label'], 0) if 'label' in row else 0
        row_id = row['id'] if 'id' in row else 0
        return {'input_ids': final_ids, 'attention_mask': final_mask, 'label': torch.tensor(label, dtype=torch.long), 'id': row_id}

# ==========================================
# 4. MAIN EXECUTION
# ==========================================
def main():
    if not os.path.exists(Config.DATA_DIR):
        print(f"CRITICAL ERROR: Data directory still not found. Please upload dataset.")
        return

    tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
    model = NarrativeModel(Config.MODEL_NAME).to(Config.DEVICE)
    
    # --- STEP 1: PRE-TRAINING ---
    print("\n🚀 STEP 1: SELF-SUPERVISED PRE-TRAINING")
    # Recursively find txt files in case they are nested
    txt_files = glob.glob(os.path.join(Config.DATA_DIR, "**/*.txt"), recursive=True)
    # Filter out non-book files if necessary
    txt_files = [f for f in txt_files if "requirements" not in f and "README" not in f]
    
    print(f"Found {len(txt_files)} books: {[os.path.basename(f) for f in txt_files]}")
    
    pretrain_ds = NarrativePretrainingDataset(txt_files, tokenizer, Config.CHUNK_SIZE)
    if len(pretrain_ds) > 0:
        print(f"Training on {len(pretrain_ds)} narrative pairs...")
        pretrain_dl = DataLoader(pretrain_ds, batch_size=32, shuffle=True, drop_last=True)
        optimizer = torch.optim.AdamW([
            {'params': model.cell.parameters(), 'lr': Config.PRETRAIN_LR},
            {'params': model.adapter.parameters(), 'lr': Config.PRETRAIN_LR}
        ])
        criterion = nn.MSELoss() 
        model.train()
        for ep in range(Config.PRETRAIN_EPOCHS):
            total_loss = 0
            pbar = tqdm(pretrain_dl, desc=f"Pre-train Ep {ep+1}")
            for batch in pbar:
                ids_t = batch['ids_t'].to(Config.DEVICE)
                ids_next = batch['ids_next'].to(Config.DEVICE)
                optimizer.zero_grad()
                pred_state, target_feat = model.forward_pretrain(ids_t, batch['mask_t'].to(Config.DEVICE), ids_next, batch['mask_next'].to(Config.DEVICE))
                loss = criterion(pred_state, target_feat)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                pbar.set_postfix({'mse': f"{loss.item():.4f}"})
        print("✅ Pre-training Complete!")
    else:
        print("⚠️ No text data found for pre-training!")

    # --- STEP 2: FINE-TUNING ---
    print("\n🚀 STEP 2: FINE-TUNING ON LABELED DATA")
    train_path = os.path.join(Config.DATA_DIR, 'train.csv')
    if os.path.exists(train_path):
        train_df = pd.read_csv(train_path)
        print(f"Loaded train.csv: {len(train_df)} rows")
        split_idx = int(0.8 * len(train_df))
        train_dl = DataLoader(LabeledNarrativeDataset(train_df.iloc[:split_idx], tokenizer), batch_size=8, shuffle=True)
        val_dl = DataLoader(LabeledNarrativeDataset(train_df.iloc[split_idx:], tokenizer), batch_size=8)
        
        optimizer = torch.optim.AdamW([
            {'params': model.classifier.parameters(), 'lr': 1e-3},
            {'params': model.cell.parameters(), 'lr': 1e-5},
            {'params': model.adapter.parameters(), 'lr': 1e-5}
        ])
        weights = torch.tensor([3.0, 1.0]).to(Config.DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weights)
        
        for ep in range(Config.FINETUNE_EPOCHS):
            model.train()
            t_loss, correct, total = 0, 0, 0
            for batch in train_dl:
                optimizer.zero_grad()
                logits = model.forward_finetune(batch['input_ids'].to(Config.DEVICE), batch['attention_mask'].to(Config.DEVICE))
                loss = criterion(logits, batch['label'].to(Config.DEVICE))
                loss.backward()
                optimizer.step()
                t_loss += loss.item()
                correct += logits.argmax(1).eq(batch['label'].to(Config.DEVICE)).sum().item()
                total += batch['label'].size(0)
            
            model.eval()
            v_corr, v_tot = 0, 0
            if len(val_dl) > 0:
                with torch.no_grad():
                    for batch in val_dl:
                        logits = model.forward_finetune(batch['input_ids'].to(Config.DEVICE), batch['attention_mask'].to(Config.DEVICE))
                        v_corr += logits.argmax(1).eq(batch['label'].to(Config.DEVICE)).sum().item()
                        v_tot += batch['label'].size(0)
                val_acc = v_corr/v_tot
            else: val_acc = 0.0
            print(f"Ep {ep+1}: Loss {t_loss:.3f} | Train Acc {correct/total:.2%} | Val Acc {val_acc:.2%}")

    # --- STEP 3: SUBMISSION ---
    print("\n🚀 STEP 3: GENERATING SUBMISSION")
    test_path = os.path.join(Config.DATA_DIR, 'test.csv')
    if os.path.exists(test_path):
        test_df = pd.read_csv(test_path)
        test_dl = DataLoader(LabeledNarrativeDataset(test_df, tokenizer), batch_size=8, shuffle=False)
        model.eval()
        predictions, ids = [], []
        with torch.no_grad():
            for batch in tqdm(test_dl):
                logits = model.forward_finetune(batch['input_ids'].to(Config.DEVICE), batch['attention_mask'].to(Config.DEVICE))
                predictions.extend(logits.argmax(1).cpu().numpy())
                ids.extend(batch['id'].numpy())
        
        final_labels = [{1: 'consistent', 0: 'contradict'}[p] for p in predictions]
        submission = pd.DataFrame({'id': ids, 'label': final_labels})
        submission.to_csv('submission.csv', index=False)
        print("✅ submission.csv saved!")
    else:
        print("⚠️ test.csv not found!")

if __name__ == "__main__":
    main()

Searching for data...
✅ FOUND DATA AT: /kaggle/input/kdshdata
Running on Device: cuda

🚀 STEP 1: SELF-SUPERVISED PRE-TRAINING
Found 2 books: ['In search of the castaways.txt', 'The Count of Monte Cristo.txt']


Token indices sequence length is longer than the specified maximum sequence length for this model (189625 > 512). Running this sequence through the model will result in indexing errors


Training on 6444 narrative pairs...


Pre-train Ep 3: 100%|██████████| 201/201 [00:50<00:00,  3.97it/s, mse=0.0005]


✅ Pre-training Complete!

🚀 STEP 2: FINE-TUNING ON LABELED DATA
Loaded train.csv: 80 rows
Ep 1: Loss 6.073 | Train Acc 62.50% | Val Acc 37.50%
Ep 2: Loss 5.446 | Train Acc 42.19% | Val Acc 31.25%
Ep 3: Loss 5.541 | Train Acc 42.19% | Val Acc 37.50%
Ep 4: Loss 4.865 | Train Acc 57.81% | Val Acc 43.75%
Ep 5: Loss 5.142 | Train Acc 59.38% | Val Acc 31.25%
Ep 6: Loss 4.615 | Train Acc 62.50% | Val Acc 31.25%
Ep 7: Loss 4.392 | Train Acc 62.50% | Val Acc 31.25%
Ep 8: Loss 4.838 | Train Acc 65.62% | Val Acc 37.50%
Ep 9: Loss 4.287 | Train Acc 60.94% | Val Acc 43.75%
Ep 10: Loss 4.868 | Train Acc 67.19% | Val Acc 50.00%

🚀 STEP 3: GENERATING SUBMISSION


100%|██████████| 8/8 [00:03<00:00,  2.02it/s]

✅ submission.csv saved!
